# 象棋 AI 训练 (Kaggle)
**步骤：**
1. 创建 Kaggle Dataset "xiangqi-data"，上传 `xqdb_masters_40711_UCI_games.pgn` 和 `best.pt`（从 Google Drive 下载）
2. 开启 GPU：Settings → Accelerator → GPU T4×2
3. Run All

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
# Kaggle input (read-only uploaded dataset)
INPUT_DIR = '/kaggle/input/xiangqi-data'
# Kaggle working (read-write output)
WORK_DIR = '/kaggle/working'
DATA_DIR = f'{WORK_DIR}/data'
CKPT_DIR = f'{WORK_DIR}/checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
# Check uploaded files
if os.path.exists(INPUT_DIR):
    print('Input files:', os.listdir(INPUT_DIR))
else:
    print('ERROR: Add dataset "xiangqi-data" with PGN + best.pt')

In [ ]:
%%writefile game.py
from __future__ import annotations
from dataclasses import dataclass
from enum import IntEnum
from typing import Optional
class PieceType(IntEnum):
    GENERAL=0;ADVISOR=1;ELEPHANT=2;HORSE=3;CHARIOT=4;CANNON=5;SOLDIER=6
class Color(IntEnum):
    RED=0;BLACK=1
    def opposite(self): return Color.BLACK if self==Color.RED else Color.RED
@dataclass
class Piece:
    type:PieceType;color:Color
@dataclass
class Move:
    from_pos:tuple;to_pos:tuple;captured_piece:Optional[Piece]=None
_O=((-1,0),(1,0),(0,-1),(0,1));_D=((-1,-1),(-1,1),(1,-1),(1,1))
_HM=((-2,-1,-1,0),(-2,1,-1,0),(-1,-2,0,-1),(-1,2,0,1),(1,-2,0,-1),(1,2,0,1),(2,-1,1,0),(2,1,1,0))
_EM=((-2,-2,-1,-1),(-2,2,-1,1),(2,-2,1,-1),(2,2,1,1))
_I=((0,0,4,1),(0,1,3,1),(0,2,2,1),(0,3,1,1),(0,4,0,1),(0,5,1,1),(0,6,2,1),(0,7,3,1),(0,8,4,1),(2,1,5,1),(2,7,5,1),(3,0,6,1),(3,2,6,1),(3,4,6,1),(3,6,6,1),(3,8,6,1),(6,0,6,0),(6,2,6,0),(6,4,6,0),(6,6,6,0),(6,8,6,0),(7,1,5,0),(7,7,5,0),(9,0,4,0),(9,1,3,0),(9,2,2,0),(9,3,1,0),(9,4,0,0),(9,5,1,0),(9,6,2,0),(9,7,3,0),(9,8,4,0))
class ChineseChessGame:
    __slots__=('board','current_player','move_history','_rp','_bp','_gp')
    def __init__(s):s.board={};s.current_player=Color.RED;s.move_history=[];s._rp=set();s._bp=set();s._gp={};s.reset()
    def reset(s):
        s.board.clear();s._rp.clear();s._bp.clear();s._gp.clear();s.current_player=Color.RED;s.move_history.clear()
        for r,c,t,co in _I:
            p=(r,c);s.board[p]=Piece(PieceType(t),Color(co));(s._rp if co==0 else s._bp).add(p)
            if t==0:s._gp[Color(co)]=p
    def _ps(s,c):return s._rp if c==Color.RED else s._bp
    def get_legal_moves(s,color=None):
        if color is None:color=s.current_player
        ps=[];b=s.board
        for pos in list(s._rp if color==Color.RED else s._bp):s._gpm(pos[0],pos[1],b[pos],ps)
        lg=[]
        for m in ps:
            s._do(m)
            if not s._chk(color) and not s._gf():lg.append(m)
            s._un(m)
        return lg
    def _gpm(s,r,c,p,ms):
        t=int(p.type);co=p.color;fp=(r,c);b=s.board
        if t==0:
            rm=7 if co==Color.RED else 0;rx=9 if co==Color.RED else 2
            for dr,dc in _O:
                nr,nc=r+dr,c+dc
                if rm<=nr<=rx and 3<=nc<=5:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==1:
            rm=7 if co==Color.RED else 0;rx=9 if co==Color.RED else 2
            for dr,dc in _D:
                nr,nc=r+dr,c+dc
                if rm<=nr<=rx and 3<=nc<=5:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==2:
            am=5 if co==Color.RED else 0;ax=9 if co==Color.RED else 4
            for dr,dc,br,bc in _EM:
                nr,nc=r+dr,c+dc
                if 0<=nr<=9 and 0<=nc<=8 and am<=nr<=ax and b.get((r+br,c+bc)) is None:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==3:
            for dr,dc,br,bc in _HM:
                nr,nc=r+dr,c+dc
                if 0<=nr<=9 and 0<=nc<=8 and b.get((r+br,c+bc)) is None:
                    tg=b.get((nr,nc))
                    if tg is None or tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
        elif t==4:
            for dr,dc in _O:
                nr,nc=r+dr,c+dc
                while 0<=nr<=9 and 0<=nc<=8:
                    tg=b.get((nr,nc))
                    if tg is None:ms.append(Move(fp,(nr,nc),None))
                    else:
                        if tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
                        break
                    nr+=dr;nc+=dc
        elif t==5:
            for dr,dc in _O:
                nr,nc=r+dr,c+dc;j=False
                while 0<=nr<=9 and 0<=nc<=8:
                    tg=b.get((nr,nc))
                    if not j:
                        if tg is None:ms.append(Move(fp,(nr,nc),None))
                        else:j=True
                    else:
                        if tg is not None:
                            if tg.color!=co:ms.append(Move(fp,(nr,nc),tg))
                            break
                    nr+=dr;nc+=dc
        elif t==6:
            fw=-1 if co==Color.RED else 1;cr=(r<=4) if co==Color.RED else(r>=5)
            nr=r+fw
            if 0<=nr<=9:
                tg=b.get((nr,c))
                if tg is None or tg.color!=co:ms.append(Move(fp,(nr,c),tg))
            if cr:
                for dc in(-1,1):
                    nc=c+dc
                    if 0<=nc<=8:
                        tg=b.get((r,nc))
                        if tg is None or tg.color!=co:ms.append(Move(fp,(r,nc),tg))
    def make_move(s,m):s._do(m);s.move_history.append(m);s.current_player=s.current_player.opposite()
    def _do(s,m):
        p=s.board[m.from_pos];del s.board[m.from_pos];ps=s._ps(p.color);ps.discard(m.from_pos)
        if m.captured_piece:s._ps(m.captured_piece.color).discard(m.to_pos)
        s.board[m.to_pos]=p;ps.add(m.to_pos)
        if p.type==PieceType.GENERAL:s._gp[p.color]=m.to_pos
    def _un(s,m):
        p=s.board[m.to_pos];ps=s._ps(p.color);del s.board[m.to_pos];ps.discard(m.to_pos)
        s.board[m.from_pos]=p;ps.add(m.from_pos)
        if m.captured_piece:s.board[m.to_pos]=m.captured_piece;s._ps(m.captured_piece.color).add(m.to_pos)
        if p.type==PieceType.GENERAL:s._gp[p.color]=m.from_pos
    def _chk(s,color):
        gp=s._gp.get(color)
        if gp is None:return True
        gr,gc=gp;opp=color.opposite();b=s.board
        for dr,dc in _O:
            nr,nc=gr+dr,gc+dc;j=False
            while 0<=nr<=9 and 0<=nc<=8:
                p=b.get((nr,nc))
                if p:
                    if not j:
                        if p.color==opp and p.type in(PieceType.CHARIOT,PieceType.GENERAL):return True
                        j=True
                    else:
                        if p.color==opp and p.type==PieceType.CANNON:return True
                        break
                nr+=dr;nc+=dc
        for dr,dc,br,bc in _HM:
            nr,nc=gr+dr,gc+dc
            if 0<=nr<=9 and 0<=nc<=8:
                p=b.get((nr,nc))
                if p and p.color==opp and p.type==PieceType.HORSE:
                    if abs(dr)==2:br2,bc2=nr+(-1 if dr>0 else 1),nc
                    else:br2,bc2=nr,nc+(-1 if dc>0 else 1)
                    if b.get((br2,bc2)) is None:return True
        if opp==Color.BLACK:
            p=b.get((gr-1,gc))
            if p and p.color==opp and p.type==PieceType.SOLDIER:return True
            for d in(-1,1):
                nc2=gc+d
                if 0<=nc2<=8:
                    p=b.get((gr,nc2))
                    if p and p.color==opp and p.type==PieceType.SOLDIER and gr>=5:return True
        else:
            p=b.get((gr+1,gc))
            if p and p.color==opp and p.type==PieceType.SOLDIER:return True
            for d in(-1,1):
                nc2=gc+d
                if 0<=nc2<=8:
                    p=b.get((gr,nc2))
                    if p and p.color==opp and p.type==PieceType.SOLDIER and gr<=4:return True
        return False
    def _gf(s):
        rp=s._gp.get(Color.RED);bp=s._gp.get(Color.BLACK)
        if not rp or not bp or rp[1]!=bp[1]:return False
        for r in range(min(rp[0],bp[0])+1,max(rp[0],bp[0])):
            if(r,rp[1])in s.board:return False
        return True
    def is_game_over(s):
        if not s.get_legal_moves(s.current_player):return True,s.current_player.opposite()
        if s._gf():return True,s.current_player
        return False,None
    def clone(s):
        g=ChineseChessGame.__new__(ChineseChessGame);g.board={p:Piece(v.type,v.color) for p,v in s.board.items()}
        g.current_player=s.current_player;g.move_history=[Move(m.from_pos,m.to_pos,Piece(m.captured_piece.type,m.captured_piece.color) if m.captured_piece else None) for m in s.move_history]
        g._rp=set(s._rp);g._bp=set(s._bp);g._gp=dict(s._gp);return g
    def get_state_key(s):return(s.current_player,tuple(sorted((p[0],p[1],v.type,v.color) for p,v in s.board.items())))

In [ ]:
%%writefile encoding.py
import torch,numpy as np
from typing import Dict,List,Tuple
ROWS=10;COLS=9;NUM_SQUARES=90;NUM_PIECE_TYPES=7;NUM_INPUT_CHANNELS=15
def _rc(r,c):return r*COLS+c
def _pos(p):return p//COLS,p%COLS
def _ib(r,c):return 0<=r<ROWS and 0<=c<COLS
def _gen():
    s=set()
    for r in range(ROWS):
        for c in range(COLS):
            fp=_rc(r,c)
            for dr,dc in[(0,1),(0,-1),(1,0),(-1,0)]:
                for st in range(1,max(ROWS,COLS)):
                    nr,nc=r+dr*st,c+dc*st
                    if _ib(nr,nc):s.add((fp,_rc(nr,nc)))
            for dr,dc in[(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
                nr,nc=r+dr,c+dc
                if _ib(nr,nc):s.add((fp,_rc(nr,nc)))
    for r,c in[(0,3),(0,5),(1,4),(2,3),(2,5),(7,3),(7,5),(8,4),(9,3),(9,5)]:
        fp=_rc(r,c)
        for dr,dc in[(-1,-1),(-1,1),(1,-1),(1,1)]:
            nr,nc=r+dr,c+dc
            if _ib(nr,nc) and((0<=nr<=2 and 3<=nc<=5)or(7<=nr<=9 and 3<=nc<=5)):s.add((fp,_rc(nr,nc)))
    for r,c in[(0,2),(0,6),(2,0),(2,4),(2,8),(4,2),(4,6),(5,2),(5,6),(7,0),(7,4),(7,8),(9,2),(9,6)]:
        fp=_rc(r,c)
        for dr,dc in[(-2,-2),(-2,2),(2,-2),(2,2)]:
            nr,nc=r+dr,c+dc
            if _ib(nr,nc) and((r<=4 and nr<=4)or(r>=5 and nr>=5)):s.add((fp,_rc(nr,nc)))
    ml=sorted(s);return ml,{m:i for i,m in enumerate(ml)}
ALL_MOVES,MOVE_TO_INDEX=_gen()
NUM_ACTIONS=len(ALL_MOVES)
def move_to_index(f,t):return MOVE_TO_INDEX[(f,t)]
def index_to_move(i):return ALL_MOVES[i]
def board_to_tensor(bs,cp=0):
    t=torch.zeros(NUM_INPUT_CHANNELS,ROWS,COLS,dtype=torch.float32)
    for pos,(co,pt) in bs.items():r,c=_pos(pos);t[co*NUM_PIECE_TYPES+pt,r,c]=1.0
    if cp==0:t[14,:,:]=1.0
    return t

In [ ]:
%%writefile model.py
import torch,torch.nn as nn,torch.nn.functional as F
from encoding import NUM_INPUT_CHANNELS,NUM_ACTIONS,ROWS,COLS
class ResBlock(nn.Module):
    def __init__(s,ch):super().__init__();s.c1=nn.Conv2d(ch,ch,3,padding=1,bias=False);s.b1=nn.BatchNorm2d(ch);s.c2=nn.Conv2d(ch,ch,3,padding=1,bias=False);s.b2=nn.BatchNorm2d(ch)
    def forward(s,x):return F.relu(s.b2(s.c2(F.relu(s.b1(s.c1(x)))))+x)
class ChineseChessNet(nn.Module):
    def __init__(s,nf=128,nb=6):
        super().__init__();s.num_filters=nf;s.bs=ROWS*COLS
        s.ci=nn.Conv2d(NUM_INPUT_CHANNELS,nf,3,padding=1,bias=False);s.bi=nn.BatchNorm2d(nf)
        s.res=nn.Sequential(*[ResBlock(nf) for _ in range(nb)])
        s.pc=nn.Conv2d(nf,2,1,bias=False);s.pb=nn.BatchNorm2d(2);s.pf=nn.Linear(2*s.bs,NUM_ACTIONS)
        s.vc=nn.Conv2d(nf,1,1,bias=False);s.vb=nn.BatchNorm2d(1);s.vf1=nn.Linear(s.bs,128);s.vf2=nn.Linear(128,1)
        for m in s.modules():
            if isinstance(m,nn.Conv2d):nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d):nn.init.constant_(m.weight,1);nn.init.constant_(m.bias,0)
            elif isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight,mode='fan_in',nonlinearity='relu')
                if m.bias is not None:nn.init.constant_(m.bias,0)
    def forward(s,x):
        o=F.relu(s.bi(s.ci(x)));o=s.res(o)
        p=F.relu(s.pb(s.pc(o)));p=s.pf(p.view(p.size(0),-1))
        v=F.relu(s.vb(s.vc(o)));v=torch.tanh(s.vf2(F.relu(s.vf1(v.view(v.size(0),-1)))))
        return p,v
    def predict(s,bt):
        w=s.training;s.eval()
        if bt.dim()==3:bt=bt.unsqueeze(0)
        with torch.no_grad():p,v=s.forward(bt)
        if w:s.train()
        return p,v
    def export_to_onnx(s,path):
        s.eval();torch.onnx.export(s,torch.zeros(1,NUM_INPUT_CHANNELS,ROWS,COLS),path,input_names=['board'],output_names=['policy','value'],dynamic_axes={'board':{0:'b'},'policy':{0:'b'},'value':{0:'b'}},opset_version=13)
def create_model(nf=128,nb=6):return ChineseChessNet(nf,nb)

In [ ]:
%%writefile trainer.py
import logging,os,random,re,gc,pickle,time,numpy as np,torch
import torch.nn.functional as F,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from model import create_model,ChineseChessNet
from encoding import board_to_tensor,move_to_index,NUM_ACTIONS,COLS,ROWS
from game import ChineseChessGame,Color,Move
logger=logging.getLogger(__name__)

def _bstate(g):return{r*COLS+c:(int(p.color),int(p.type))for(r,c),p in g.board.items()}

# === UCI PGN Parser ===
def parse_uci_pgn(filepath):
    games=[];cur={}
    with open(filepath) as f:
        for line in f:
            line=line.strip()
            if line.startswith('['):
                m=re.match(r'\[(\w+)\s+"(.*?)"\]',line)
                if m:cur[m.group(1)]=m.group(2)
            elif line:
                ms=re.sub(r'\d+\.','',line)
                mvs=[m for m in ms.split() if m not in('1-0','0-1','1/2-1/2','*')and len(m)==4]
                if mvs:cur['moves']=mvs;games.append(cur);cur={}
    return games

def uci_to_pos(m):
    return(9-int(m[1]),ord(m[0])-ord('a')),(9-int(m[3]),ord(m[2])-ord('a'))

def find_move(g,fp,tp):
    for m in g.get_legal_moves():
        if m.from_pos==fp and m.to_pos==tp:return m
    return None

def replay_uci_compact(moves):
    g=ChineseChessGame();pos=[]
    for uci in moves:
        try:fp,tp=uci_to_pos(uci)
        except:break
        bs=_bstate(g);cp=int(g.current_player)
        m=find_move(g,fp,tp)
        if m is None:break
        try:ai=move_to_index(fp[0]*COLS+fp[1],tp[0]*COLS+tp[1])
        except:break
        pos.append((bs,ai,cp));g.make_move(m)
        if g.is_game_over()[0]:break
    return pos

# === Process and save in chunks ===
def process_and_save(filepath, save_dir, chunk_size=50000):
    print('Parsing PGN...', flush=True)
    games=parse_uci_pgn(filepath)
    print(f'Found {len(games)} games', flush=True)
    os.makedirs(save_dir, exist_ok=True)
    chunk=[];ok=fail=chunk_id=total=0
    for gd in games:
        mvs=gd.get('moves',[])
        if not mvs:fail+=1;continue
        pos=replay_uci_compact(mvs)
        if len(pos)<10:fail+=1;continue
        res=gd.get('Result','*')
        val=1.0 if res=='1-0' else(-1.0 if res=='0-1' else 0.0)
        for bs,ai,cp in pos:chunk.append((bs,ai,cp,val))
        ok+=1
        if len(chunk)>=chunk_size:
            path=os.path.join(save_dir,f'chunk_{chunk_id:03d}.pkl')
            with open(path,'wb') as f:pickle.dump(chunk,f,protocol=4)
            total+=len(chunk)
            print(f'  Chunk {chunk_id}: {len(chunk)} pos (total: {total})', flush=True)
            chunk=[];chunk_id+=1;gc.collect()
        if ok%2000==0:print(f'  Games: {ok} ok, {fail} fail', flush=True)
    if chunk:
        path=os.path.join(save_dir,f'chunk_{chunk_id:03d}.pkl')
        with open(path,'wb') as f:pickle.dump(chunk,f,protocol=4)
        total+=len(chunk);chunk_id+=1
    print(f'Done: {ok} games, {chunk_id} chunks, {total} positions', flush=True)
    meta={'num_chunks':chunk_id,'total':total}
    with open(os.path.join(save_dir,'meta.pkl'),'wb') as f:pickle.dump(meta,f)
    return total

# === Dataset ===
class CompactDS(Dataset):
    def __init__(s,data):s.data=data
    def __len__(s):return len(s.data)
    def __getitem__(s,i):
        bs,ai,cp,val=s.data[i]
        board=board_to_tensor(bs,cp)
        policy=torch.zeros(NUM_ACTIONS,dtype=torch.float32);policy[ai]=1.0
        value=val if cp==0 else -val
        return board,policy,torch.tensor(value,dtype=torch.float32)

# === Training ===
def train_from_chunks(model, data_dir, epochs=30, bs=512, lr=0.001, device=None, ckpt_dir='./ckpt', start_epoch=1):
    device=device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(ckpt_dir,exist_ok=True)
    with open(os.path.join(data_dir,'meta.pkl'),'rb') as f:meta=pickle.load(f)
    nc=meta['num_chunks']
    print(f'{nc} chunks, {meta["total"]} positions. Training chunk-by-chunk.', flush=True)
    # Use last chunk as validation
    with open(os.path.join(data_dir,f'chunk_{nc-1:03d}.pkl'),'rb') as f:val_data=pickle.load(f)
    vds=CompactDS(val_data);del val_data;gc.collect()
    print(f'Val set: {len(vds)} positions', flush=True)
    model.to(device)
    opt=optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    sch=optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=0.5,patience=3)
    best=float('inf')
    train_chunks=list(range(nc-1))
    for ep in range(start_epoch,epochs+1):
        model.train();random.shuffle(train_chunks)
        tp=tv=tc=ts=nb=0;t0=time.time()
        for ci_idx,ci in enumerate(train_chunks):
            with open(os.path.join(data_dir,f'chunk_{ci:03d}.pkl'),'rb') as f:chunk=pickle.load(f)
            random.shuffle(chunk)
            tds=CompactDS(chunk);tl=DataLoader(tds,batch_size=bs,shuffle=True,num_workers=2,pin_memory=True)
            for b,p,v in tl:
                b,p,v=b.to(device),p.to(device),v.to(device)
                pl,pv=model(b);pv=pv.squeeze(-1)
                ploss=-torch.sum(p*F.log_softmax(pl,1),1).mean();vloss=F.mse_loss(pv,v)
                loss=ploss+vloss;opt.zero_grad();loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
                tp+=ploss.item();tv+=vloss.item();nb+=1;ts+=b.size(0)
                tc+=(pl.argmax(1)==p.argmax(1)).sum().item()
            del chunk,tds;gc.collect()
            if ci_idx==0 or (ci_idx+1)%5==0:
                elapsed=time.time()-t0
                print(f'  Ep{ep} chunk {ci_idx+1}/{len(train_chunks)} loss={tp/max(1,nb)+tv/max(1,nb):.4f} acc={tc/max(1,ts)*100:.1f}% [{elapsed:.0f}s]', flush=True)
        model.eval();vtp=vtv=vtc=vts=vnb=0
        vl2=DataLoader(vds,batch_size=bs,shuffle=False,num_workers=2)
        with torch.no_grad():
            for b,p,v in vl2:
                b,p,v=b.to(device),p.to(device),v.to(device)
                pl2,pv2=model(b);pv2=pv2.squeeze(-1)
                vtp+=(-torch.sum(p*F.log_softmax(pl2,1),1).mean()).item()
                vtv+=F.mse_loss(pv2,v).item();vnb+=1;vts+=b.size(0)
                vtc+=(pl2.argmax(1)==p.argmax(1)).sum().item()
        vl=vtp/max(1,vnb)+vtv/max(1,vnb);sch.step(vl)
        print(f'Ep {ep}/{epochs}: train={tp/max(1,nb)+tv/max(1,nb):.4f} acc={tc/max(1,ts)*100:.1f}% | val={vl:.4f} acc={vtc/max(1,vts)*100:.1f}%', flush=True)
        if vl<best:
            best=vl;torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'val_loss':vl},os.path.join(ckpt_dir,'best.pt'))
            print(f'  Saved best ({vl:.4f})', flush=True)
    return model

## 处理数据 + 训练

In [ ]:
import os
from trainer import process_and_save

# Find PGN in input dataset
PGN_PATH = None
for f in os.listdir(INPUT_DIR):
    if f.endswith('.pgn'):
        PGN_PATH = os.path.join(INPUT_DIR, f)
        break
if PGN_PATH is None:
    raise FileNotFoundError('No .pgn file in dataset!')
print(f'PGN: {PGN_PATH}')

META = f'{DATA_DIR}/meta.pkl'
if os.path.exists(META):
    import pickle
    with open(META,'rb') as f: m=pickle.load(f)
    print(f'Data already processed: {m["total"]} positions in {m["num_chunks"]} chunks')
else:
    process_and_save(PGN_PATH, DATA_DIR)

In [ ]:
from trainer import train_from_chunks
from model import create_model
import torch, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')
model = create_model(128, 6)

# Load checkpoint from input dataset (uploaded from Colab/Drive) or from working dir
BEST_IN = os.path.join(INPUT_DIR, 'best.pt')
BEST_OUT = f'{CKPT_DIR}/best.pt'
start_ep = 1

ckpt_path = BEST_OUT if os.path.exists(BEST_OUT) else (BEST_IN if os.path.exists(BEST_IN) else None)
if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    start_ep = ckpt.get('epoch', 0) + 1
    print(f'Resumed from epoch {start_ep-1} (val_loss={ckpt["val_loss"]:.4f}) via {ckpt_path}')
else:
    print('Training from scratch')

model = train_from_chunks(model, DATA_DIR, epochs=30, bs=512, lr=0.001, device=device, ckpt_dir=CKPT_DIR, start_epoch=start_ep)
print('Training complete!')

## 下载模型
训练完成后，在右侧 Output 面板下载 `checkpoints/best.pt`

In [ ]:
# Export ONNX to output
from model import create_model
import torch
model = create_model(128, 6)
ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.cpu().eval()
onnx_path = f'{CKPT_DIR}/best.onnx'
model.export_to_onnx(onnx_path)
print(f'ONNX exported to {onnx_path}')
print(f'Download from Output panel: checkpoints/best.pt and checkpoints/best.onnx')